In [12]:
import pyranges as pr
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [13]:
df_RPKM= pd.read_csv('45944_mRNA_S15_L003_R1_001_qexpr_rpkm-Copy1.txt', sep='\t', header=None, 
                     names=["gene_id", "length", "footprint_reads", "RPM", "length_kb", "RPKM"]) #loading RPKM file

In [14]:
def get_rpkm(df, gene_id):
    return df.loc[df['gene_id'] == gene_id, 'RPKM'].iloc[0]

cyc8_RPKM = get_rpkm(df_RPKM, "YBR112C")

In [15]:
cyc8_RPKM

'287.1369708818336'

In [16]:
path = ("/mnt/unallab/jleslie/20250827_ribosome_profiling_snf5_cyc8_ade2/28_29merAnalysis/cyc8analysis/45944_monosome_S11_L003_R1_001_Aligned_unique_sorted_rev_YBR112C.bed")
#this bed file was made by extracting the CYC8 coordinates from the A-site assigned .wig file for each sample
gr = pr.read_bed(path)
df = gr.df

In [17]:
region_start = 462872
region_end = 465770
num_positions = region_end - region_start

#normalize by RPKM
df['Score'] = df['Name'].div(float(cyc8_RPKM))
df['Score'] = df['Score'].fillna(0)

#build score dict with all positions initialized to 0
score_dict_rev = {i: 0 for i in range(num_positions)}

#fill in real scores and reverse for plotting
for index, row in df.iterrows():
    pos = row['Start'] - region_start
    if 0 <= pos < num_positions:
        rev_pos = (num_positions - 1) - pos
        score_dict_rev[rev_pos] = row['Score']

score_list_cyc8 = [score_dict_rev[i] for i in range(num_positions)]

df_cyc8 = pd.DataFrame({
    "Position": list(range(num_positions)),
    "Score": score_list_cyc8
})

df_cyc8.to_csv('041526_45944_normalized_asiteoccupancy_monosomes_cyc8.csv', index=False)
print(f"Length: {len(score_list_cyc8)}")
df_cyc8

Length: 2898


,Position,Score
0,0,0.000000
1,1,0.000000
2,2,0.010448
3,3,0.041792
4,4,0.000000
...,...,...
2893,2893,0.017413
2894,2894,0.013931
2895,2895,0.003483
2896,2896,0.000000


In [18]:
#calculate frame periodicity 
score_list_rev = list(score_dict_rev.values())

firstframe  = score_list_rev[0::3]
secondframe = score_list_rev[1::3]
thirdframe  = score_list_rev[2::3]

total = sum(score_list_rev)
print({
    "first":  sum(firstframe)  / total,
    "second": sum(secondframe) / total,
    "third":  sum(thirdframe)  / total
})

{'first': 0.868756835581477, 'second': 0.09259934378417754, 'third': 0.03864382063434167}


In [25]:
#frame periodicity before and after premature stop (premature stop included in 'after stop')
before_stop = score_list_rev[:1941]
after_stop = score_list_rev[1941:]
len(after_stop) #checking length
len(before_stop) #checking length

def frame(x):
    firstframe  = x[0::3]
    secondframe = x[1::3]
    thirdframe  = x[2::3]
    total = sum(x)
    
    return {
        "first":  sum(firstframe)  / total,
        "second": sum(secondframe) / total,
        "third":  sum(thirdframe)  / total
    }

frames_before_45944 = frame(before_stop)
frames_after_45944 = frame(after_stop)

print(frames_before_45944)
print(frames_after_45944)

{'first': 0.8764258555133047, 'second': 0.08555133079847885, 'third': 0.0380228136882127}
{'first': 0.6902654867256638, 'second': 0.2566371681415931, 'third': 0.0530973451327434}


In [30]:
#counts for all frames
score_list_rev = score_list_cyc8

cdsstart = score_list_rev[0]
cdsend = score_list_rev[1944]

utr5len = 0
cdslen = 1944 #include premature stop in 'cds'
utr3len = len(score_list_rev) - 1944 
mrnalen = len(score_list_rev)

#counts for all frames
utr3Counts = sum(score_list_rev[cdslen:])
cdsCounts = sum(score_list_rev[:(cdslen-1)])
mrnaCounts = cdsCounts+utr3Counts

#rpk for all frames
utr3Density_rpm = (utr3Counts/len(score_list_rev[cdslen:]))*1000
cdsDensity_rpk = (cdsCounts/len(score_list_rev[:(cdslen-1)]))*1000
mrnaDensity_rpk = ((cdsCounts+utr3Counts)/len(score_list_rev))*1000

#rrts for all frames
RRTS = utr3Density_rpk/cdsDensity_rpk
print(RRTS)

0.08670049968645044
